# TP3: Bag of Words (BoW)

### 1. Implement a bag of words algorithm with Python

In [16]:
import re

s1 = "Welcome to NLP Learning, Now start learning"
s2 = "Learning is a good practice"

# lowercase and split
lw1 = re.findall(r'\w+|[,]', s1.lower())
lw2 = re.findall(r'\w+|[,]', s2.lower())

# commbined and remove duplicates
combined_words_list = lw1 + lw2
combined_words_list.pop(7)
combined_words_list.pop(7)

# remove stopwords
stopwords = ["to", "is", "a", ","]
remove_stopwords = [w for w in combined_words_list if w not in stopwords]

# count how many times each word in remove_stopwords appear in each sentence
counts_s1 = [lw1.count(w.lower()) for w in remove_stopwords]
counts_s2 = [lw2.count(w.lower()) for w in remove_stopwords]

# print the result
print(lw1)
print(lw2)
print(combined_words_list)
print(remove_stopwords)
print(counts_s1)
print(counts_s2)

['welcome', 'to', 'nlp', 'learning', ',', 'now', 'start', 'learning']
['learning', 'is', 'a', 'good', 'practice']
['welcome', 'to', 'nlp', 'learning', ',', 'now', 'start', 'is', 'a', 'good', 'practice']
['welcome', 'nlp', 'learning', 'now', 'start', 'good', 'practice']
[1, 1, 2, 1, 1, 0, 0]
[0, 0, 1, 0, 0, 1, 1]


### 2. Implement Bag of Words using SKLEARN

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

sentence1 ="This is a good job. I will not miss it for anything"
sentence2 = "This is not good at all"

# put both sentence into a list
docs = [sentence1, sentence2]

# remove stopwords
stopwords = ['all', 'anything', 'i', 'it', 'not', 'this', 'for', 'at', 'is', 'will']
clean_docs = []

for text in docs:
    words = text.lower().split()
    filtered = [w for w in words if w not in stopwords]
    clean_docs.append(" ".join(filtered))

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(clean_docs)

print(vectorizer.get_feature_names_out())
print(X.toarray())

['good' 'job' 'miss']
[[1 1 1]
 [1 0 0]]


### 3. Implement Bag of words using NLTK
Task:
- Importing the necessary libraries from NLTK.
- Define a list of sample documents.
- Tokenize the documents into words and convert them to lowercase.
- Remove stopwords and punctuation from the tokens.
- Create a vocabulary by collecting all unique words from the processed documents.
- Initialize a BoW dictionary with word counts, setting the initial count for each word to 0.
- Iterate through the filtered tokens and increment the count for each word in the BoW dictionary.
- Print the BoW representation, which shows the word counts for each word in the vocabulary.

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import string

# nltk.download('punkt')
# nltk.download('stopwords')

docs = ["I love natural language processing.", "Text classification is an important NLP task.", "NLTK provides useful tools for NLP."]

stopwords = set(stopwords.words('english'))
punctuation = set(string.punctuation)

cleaned_docs = []
for doc in docs:
    # tokenize
    tokens = word_tokenize(doc.lower())
    # remove stopwords and punctuation
    filtered_tokens = [t for t in tokens if t not in stopwords and t not in punctuation]
    cleaned_docs.append(filtered_tokens)

all_tokens = [word for doc in cleaned_docs for word in doc]
bow = dict(Counter(all_tokens))

print(bow)

{'love': 1, 'natural': 1, 'language': 1, 'processing': 1, 'text': 1, 'classification': 1, 'important': 1, 'nlp': 2, 'task': 1, 'nltk': 1, 'provides': 1, 'useful': 1, 'tools': 1}


### 4. Classify movie review is posiHve or negaHve using Bag of words for pre-processing the text (from Sklearn) and apply with any models (RF, DT)

Dataset:
Link = https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?resource=download
- This data consists of two columns. - review - sentiment
- Reviews are the statements given by users watching the movie.
- sentiment feature tells whether the given review is positive or negative.

In [73]:
import pandas as pd

df = pd.read_csv("./IMDB Dataset.csv")

print(f"First 5 rows: {df.head()}\n")
print(f"Shape of the dataset: {df.shape}")

First 5 rows:                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Shape of the dataset: (50000, 2)


In [ ]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stopwords = set(stopwords.words("english"))
punctuation = str.maketrans('', '', string.punctuation)

def clean_text(text):
    # remove html tag
    text = re.sub(r'<.*?>', '', text)
    # remove punctuation
    text = text.translate(punctuation)
    # tokenize word
    tokens = word_tokenize(text.lower())
    # remove stopwords
    tokens = [word for word in tokens if word not in stopwords]

    return ' '.join(tokens)

df['clean'] = df['review'].apply(clean_text)

In [ ]:
# map positive to 1 and negative to 0
df['sentiment'] = df['sentiment'].map(lambda x: 1 if x == 'positive' else 0)

In [81]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['clean'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"Classification Report for Random Forest:\n\n {classification_report(y_test, y_pred_rf)}")

print("="*50)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print(f"Classification Report for Decision Tree:\n\n {classification_report(y_test, y_pred_dt)}")


Classification Report for Random Forest:

               precision    recall  f1-score   support

           0       0.85      0.87      0.86      4961
           1       0.87      0.86      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

Classification Report for Decision Tree:

               precision    recall  f1-score   support

           0       0.73      0.73      0.73      4961
           1       0.74      0.73      0.73      5039

    accuracy                           0.73     10000
   macro avg       0.73      0.73      0.73     10000
weighted avg       0.73      0.73      0.73     10000

